# TP 2 — Comparer ML et deep learning sur Tiny Shakespeare

Le but de ce TP est de comparer les performances du machine learning et du deep learning sur une tâche d'identification du locuteur dans le dataset Tiny Shakespeare.

Je cherche ce que le deep learning apporte à l’identification du locuteur. Je pars du TP1 et de mes analyses pour construire un pipeline de deep learning.

Je suis les quatre étapes du [cours, p. 65](Cours/NLP_slides.pdf#page=65) : nettoyage et prétraitement, représentation, comparaison ML/DL, évaluation.


## 0. Mon problème et les acquis du TP1

Dans la formulation du problème, je prédis une étiquette de locuteur à partir du seul texte d’une réplique, parmi les 10 classes retenues. Je garde les répliques d’au moins 10 mots, puis les locuteurs les plus fréquents (expliqué dans le TP1).

| Fait observé dans le TP1 | Ce que je dois examiner dans le TP2 |
|---|---|
| 962 répliques, de 78 à 129 par classe | Je surveille le sur-apprentissage sur ce petit jeu, modérément déséquilibré (choisis exprès pour réduire le déséquilibre d'exemples par classe). |
| 9 confusions JULIET–ROMEO et 7 de GLOUCESTER vers DUKE VINCENTIO sur le test | Je relis les erreurs pour chercher des indices communs. |
| `kate`, `paulina`, `camillo` et `romeo` ont des poids élevés | Je distingue les indices liés aux personnages d’une reconnaissance du style. |
| LinearSVC : F1 macro de 0,482 ± 0,035 en validation croisée, contre 1 sur les trains des folds | Je compare la généralisation, pas seulement l’ajustement au train. |

> *Mes hypothèses :*
>
> - Les indices lexicaux peuvent suffire à une partie des prédictions, même avec un modèle simple.
> - L’ordre des mots peut apporter du contexte supplémentaire. Le TF–IDF du TP1 capture déjà de l’ordre local avec ses bigrammes.
> - Un réseau peut mémoriser les répliques sans mieux classer de nouveaux exemples.

Je ne sais pas encore quelle approche convient le mieux.


### Mon environnement

Je sélectionne `.venv/bin/python` comme kernel. J’installe les dépendances dans le terminal :

```sh
uv pip install --python .venv/bin/python -r requirements.txt
```

> *Mes choix :*
>
> - Je choisis PyTorch pour coder explicitement le forward, la loss, le backward et la mise à jour des poids.
> - J’utilise le GPU Apple avec MPS s’il est disponible, sinon le CPU (meilleur que colab). 
> - Je fixe la seed à 42 et je la réinitialiserai avant chaque essai. Ca contrôle le hasard, sans garantir des résultats identiques entre devices ou versions.


In [2]:
import platform
import random
import subprocess
import sys
from importlib.metadata import version

import numpy as np
import torch

# Random seed for reproducibility
RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')

# Print environment information
print(f'Python : {platform.python_version()} — {sys.executable}')
for paquet in ('numpy', 'pandas', 'matplotlib', 'scikit-learn', 'ipykernel', 'torch'):
    print(f'{paquet} : {version(paquet)}')

# Print system information
print(f'Système : {platform.platform()} — {platform.machine()}')
if sys.platform == 'darwin':
    processeur, ram = subprocess.check_output(
        ['sysctl', '-n', 'machdep.cpu.brand_string', 'hw.memsize'], text=True,
    ).splitlines()
    print(f'Processeur : {processeur} — RAM : {int(ram) / 2**30:g} Gio')

# Final infos
print(f'Device : {device} — Seed : {RANDOM_STATE}')


Python : 3.14.6 — /Users/hopy/Documents/ING3_GIA/ING3_IA_GB_S1/NLP/.venv/bin/python
numpy : 2.5.3
pandas : 3.0.5
matplotlib : 3.11.1
scikit-learn : 1.9.0
ipykernel : 7.3.0
torch : 2.14.0
Système : macOS-26.5-arm64-arm-64bit-Mach-O — arm64
Processeur : Apple M3 Max — RAM : 36 Gio
Device : mps — Seed : 42


## 1. Nettoyage et prétraitement

### 1.1 Retrouver les mêmes répliques

Je reprends le chargement et le parser du TP1 dans ce notebook, sans dépendre de son état d’exécution. Le nom en tête du bloc fournit l’étiquette ; il ne doit pas rester dans l’entrée du modèle.

Je conserve les pronoms, négations et noms présents dans la réplique : « I », « not » ou « Romeo » peuvent être des indices. Je vérifie ce que ma tokenisation garde, notamment les apostrophes.


In [ ]:
# TODO : charger Tiny Shakespeare et reprendre parser_repliques du TP1.
# TODO : filtrer à 10 mots avant de sélectionner les 10 locuteurs ; conserver les indices.
# TODO : vérifier les 962 lignes, les labels, les doublons, les longueurs et quelques répliques.


### 1.2 Séparer apprentissage, arrêt et évaluation

Je reproduis le découpage stratifié du TP1 : 769 répliques de développement et les mêmes 193 répliques de test, avec la seed 42.

Sur les 769 répliques, je prépare `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)`. Tous les modèles utilisent les mêmes cinq folds.

| Ensemble dans un fold | Son rôle |
|---|---|
| Train du fold | Je l’utilise pour ajuster le modèle ML et son TF–IDF. |
| 80 % du train du fold, en DL | J’y apprends le vocabulaire et les paramètres du réseau. |
| 20 % restants, stratifiés, en DL | Je choisis le checkpoint par early stopping, selon le F1 macro interne. |
| Validation externe du fold | Je mesure le modèle retenu ; elle ne règle ni le prétraitement ni l’arrêt. |

Je réutilise aussi le même découpage interne pour les trois réseaux. Je rapporte leurs effectifs : la partie utilisée pour les gradients est plus petite que le train du modèle ML. Le test reste réservé à la comparaison finale.


In [ ]:
# TODO : reproduire et conserver les indices train/test du TP1 ; vérifier 769/193 et leur disjonction.
# TODO : figer les cinq folds, puis les splits internes 80/20 stratifiés avec la seed 42.
# TODO : vérifier les classes présentes et l’absence de répliques identiques entre ensembles.


## 2. Représentation

### 2.1 Ma référence TF–IDF

Je garde les paramètres du TP1 : `lowercase=True`, `ngram_range=(1, 2)`, `min_df=2`, `max_df=1.0`, `stop_words=None`, `sublinear_tf=True`.

Le `Pipeline` réapprend le vocabulaire et l’IDF sur le train de chaque fold. Je ne réutilise pas la matrice déjà ajustée dans le TP1.


In [ ]:
# TODO : préparer un Pipeline TF–IDF avec les paramètres du TP1.
# TODO : contrôler la taille du vocabulaire et les matrices dans chaque fold.


### 2.2 Des tokens aux embeddings

Un embedding associe un vecteur dense à un token. Je l’apprends ici pendant la classification. Par exemple, `romeo` reçoit un identifiant, puis une ligne de la matrice d’embeddings. [Cours, p. 55](Cours/NLP_slides.pdf#page=55).

Je garde le même tokenizer et le même vocabulaire pour les trois réseaux d’un fold. `<UNK>` représente un token absent du train interne ; `<PAD>` complète les séquences d’un batch. Je mesure le taux de tokens inconnus et le taux de troncature.

Je choisis la longueur maximale à partir du train interne, puis je l’applique aux validations. Je justifierai la dimension des embeddings lors de l’implémentation.

La moyenne des embeddings ignore l’ordre. Exemple construit : « Romeo loves Juliet » et « Juliet loves Romeo » ont la même moyenne. Un LSTM ou un GRU peut les représenter différemment. J’exclus le padding de la moyenne et je récupère l’état récurrent après le dernier token réel.


In [ ]:
# TODO : définir le tokenizer commun et le vocabulaire du train interne avec <UNK> et <PAD>.
# TODO : justifier dimension et longueur maximale ; mesurer les inconnus et la troncature.
# TODO : préparer les batches, les longueurs, les masques et un encodage commun des 10 labels.
# TODO : vérifier qu’ajouter du padding ne change pas la représentation d’une réplique.


## 3. Comparaison entre modèles ML et DL

### 3.1 Préparer l’entraînement commun

Je compare la classe majoritaire, LinearSVC, puis MLP, LSTM et GRU. La cible reste une classe parmi dix : j’utilise la cross-entropy pour entraîner les réseaux et le F1 macro pour les comparer. [Cours, p. 50–54](Cours/NLP_slides.pdf#page=50).

Je contrôle d’abord un batch : entrées, labels, dix scores par réplique et loss finie. Le forward calcule les scores, le backward calcule les gradients, puis l’optimiseur met les poids à jour.

Je teste Adam comme point de départ du cours. Je justifierai le learning rate, le batch size, la régularisation et la patience de l’early stopping. Je garde le même budget maximal d’epochs et les mêmes règles d’arrêt pour les réseaux ; je trace loss et F1 macro sur train et validation interne.


In [ ]:
# TODO : préparer une boucle commune : forward, cross-entropy, backward et mise à jour des poids.
# TODO : contrôler les dimensions, la loss et les gradients sur un batch.
# TODO : enregistrer les courbes et restaurer le checkpoint au meilleur F1 macro interne.
# TODO : compter les paramètres entraînables et chronométrer l’entraînement sur le même matériel.


### 3.2 Baseline majoritaire

> *Mon hypothèse :* Prédire la classe la plus fréquente du train donne un repère faible mais nécessaire. Un modèle textuel doit apporter un gain mesuré par rapport à ce repère.


In [ ]:
# TODO : évaluer DummyClassifier(strategy="most_frequent") sur les cinq folds.
# TODO : conserver les prédictions, l’accuracy et le F1 macro de chaque fold.


> *Mes observations :*
>
> *Ma conclusion :*


### 3.3 TF–IDF + LinearSVC

> *Mon hypothèse :* Sur ce petit jeu, les indices lexicaux peuvent rester compétitifs face aux réseaux. LinearSVC convient aux vecteurs TF–IDF creux du TP1. [Cours, p. 33](Cours/NLP_slides.pdf#page=33).

Je reprends `C=1`, `class_weight=None`, `max_iter=5000` et `random_state=42`. Je recalcule les scores dans le protocole du TP2.


In [ ]:
# TODO : évaluer le Pipeline TF–IDF + LinearSVC sur les folds communs.
# TODO : conserver les prédictions par indice, les scores train/validation et les temps.


> *Mes observations :*
>
> *Ma conclusion :*


### 3.4 Embeddings + moyenne + MLP

> *Mon hypothèse :* Un petit MLP peut apprendre des combinaisons utiles d’indices après la moyenne des embeddings. Je teste cette représentation sans lui fournir l’ordre des mots.

Je prévois une couche cachée avec ReLU, puis dix scores de classe. Je justifierai sa largeur selon le nombre de paramètres et le sur-apprentissage observé. [Cours, p. 49–50](Cours/NLP_slides.pdf#page=49).


In [ ]:
# TODO : construire embeddings, moyenne masquée, couche cachée ReLU et sortie à 10 classes.
# TODO : entraîner dans chaque fold avec la boucle commune et l’early stopping interne.
# TODO : conserver les prédictions externes, les courbes, les meilleures epochs et les temps.


> *Mes observations :*
>
> *Ma conclusion :*


### 3.5 Embeddings + LSTM

> *Mon hypothèse :* Lire la séquence peut apporter des indices que la moyenne perd. Je teste si cela corrige certaines erreurs, malgré le faible nombre de répliques.

Le LSTM conserve une mémoire `c` en plus de son état caché `h`. Ses portes contrôlent ce qu’il garde et transmet. Je commence avec une couche unidirectionnelle, puis une tête de classification linéaire à dix sorties. [Cours, p. 56–58](Cours/NLP_slides.pdf#page=56).


In [ ]:
# TODO : construire embeddings, LSTM à une couche et tête linéaire à 10 classes.
# TODO : utiliser les longueurs réelles pour exclure le padding de l’état retenu.
# TODO : évaluer avec la boucle, les splits et les règles d’arrêt communs.


> *Mes observations :*
>
> *Ma conclusion :*


### 3.6 Embeddings + GRU : comparaison avec le LSTM

> *Mon hypothèse :* Le GRU peut suffire pour nos répliques avec moins de paramètres qu’un LSTM de même largeur. Je compare les résultats et le coût avant de choisir.

Le GRU utilise un état unique et deux portes. Je garde une couche unidirectionnelle, la même dimension d’embedding, la même largeur cachée et la même tête que le LSTM. Tokenisation, vocabulaire, initialisation des embeddings et réglages d’entraînement restent communs ; les poids sont appris séparément.

Une largeur égale ne donne pas un nombre de paramètres égal. Je compte donc les paramètres et compare F1 macro, dispersion, temps et écart train/validation. [Cours, p. 59–60](Cours/NLP_slides.pdf#page=59).


In [ ]:
# TODO : remplacer le LSTM par un GRU en conservant les autres choix de cette comparaison.
# TODO : vérifier les paramètres entraînables et le traitement du padding.
# TODO : conserver prédictions, courbes, meilleures epochs et temps sur les mêmes folds.


> *Mes observations :*
>
> *Ma conclusion :*


## 4. Évaluation, erreurs et conclusions

### 4.1 Comparer la généralisation

Je rassemble l’accuracy et le F1 macro de chaque fold, puis leur moyenne et leur écart-type. Le F1 macro donne le même poids aux dix classes. La dispersion décrit ces cinq scores ; elle ne constitue pas un test de significativité.

Je compare aussi les métriques par classe, le nombre de paramètres et le temps total, early stopping compris. Je conserve les prédictions de validation externe par indice pour comparer les modèles sur les mêmes répliques. [Cours, p. 48](Cours/NLP_slides.pdf#page=48).


In [ ]:
# TODO : afficher les scores par fold, les moyennes, les écarts-types, les paramètres et les temps.
# TODO : tracer et commenter les écarts train/validation à partir des courbes enregistrées.
# TODO : afficher précision, rappel, F1 et supports par classe, puis les matrices de confusion.


### 4.2 Revenir aux répliques

Je lis les erreurs communes et les désaccords entre ML et DL. Je suis particulièrement JULIET–ROMEO et GLOUCESTER–DUKE VINCENTIO, sans supposer que les erreurs du TP1 se reproduisent.

Quelles erreurs le DL corrige-t-il ? Lesquelles introduit-il ? Les noms, le vocabulaire partagé, la longueur ou les tokens inconnus donnent-ils une explication vérifiable ?


In [ ]:
# TODO : afficher des répliques entières avec le label réel et les prédictions des modèles.
# TODO : examiner cinq cas concrets, les confusions ciblées et les supports correspondants.


> *Mes observations :*
>
> *Ma conclusion :*


### 4.3 Expériences ciblées, si les résultats le justifient

> *Mon hypothèse sur l’ordre :* Mélanger les mots d’une réplique peut dégrader un modèle qui utilise leur ordre. Je garde les mêmes mots et applique une perturbation reproductible au train et aux validations.
>
> *Mon hypothèse sur les noms :* Masquer une liste de noms choisie uniquement sur le train peut réduire certains indices liés aux personnages. `kate` et `paulina` motivent cette question depuis le TP1 ; ils ne fixent pas une liste à partir du test.

Je change un seul facteur et je réentraîne dans les folds, avant l’évaluation finale. Une comparaison MLP/LSTM/GRU ne suffit pas à attribuer un gain à l’ordre ; masquer quelques noms ne démontre pas non plus que le signal restant est du style.


In [ ]:
# TODO : choisir une hypothèse à tester sur les folds de développement, avant le test final.
# TODO : figer la perturbation, garder le reste du protocole et comparer au modèle non modifié.


> *Mes observations :*
>
> *Ma conclusion :*


### 4.4 Figer les choix et évaluer sur le test

Je choisis les configurations selon le F1 macro moyen en validation, en examinant leur dispersion et leur coût. Pour chaque réseau retenu, je fixe le nombre d’epochs à la médiane de ses cinq meilleures epochs internes. Je reconstruis ensuite son vocabulaire sur les 769 répliques et je réentraîne depuis une nouvelle initialisation, sans utiliser le test pour m’arrêter.

Je réajuste aussi les références ML sur les 769 répliques. Je compare les choix figés sur les mêmes 193 répliques de test, puis j’analyse leurs erreurs sans relancer de réglage à partir de ces scores.

Ce test a déjà servi dans le TP1 : cette comparaison reste exploratoire. Les hypothèses issues de ses erreurs ne transforment pas ce jeu en test indépendant.


In [ ]:
# TODO : figer les configurations et les epochs médianes ; reconstruire les modèles sur les 769 répliques.
# TODO : évaluer les choix figés sur les 193 répliques réservées et conserver leurs prédictions.
# TODO : comparer scores, matrices de confusion, erreurs et coût dans un tableau final.


> *Mes observations :*
>
> *Ma conclusion :*


### 4.5 Mon journal d’expériences

Je remplis une ligne après chaque essai. Je garde une hypothèse même si le résultat ne la confirme pas.

| Expérience | Hypothèse | Changement | Observation chiffrée | Décision |
|---|---|---|---|---|
| | | | | |

### 4.6 Ce que je peux conclure

Quel modèle répond le mieux au problème, compte tenu des scores et du coût ? Quelles hypothèses mes expériences soutiennent-elles ? Qu’est-ce qui reste incertain sur le contexte, le style et la généralisation ?

> *Ma conclusion :*
